In [1]:
import os

# Funciones Definidas por el Usuario (UDF) en Apache Spark

Aunque Apache Spark proporciona un catálogo enorme de funciones integradas altamente optimizadas para los casos de uso más comunes, siempre habrá escenarios particulares en los que ninguna de ellas pueda resolver la lógica exacta que nuestro negocio requiere.

Para estos casos, Spark ofrece un mecanismo flexible que permite escribir **Funciones Definidas por el Usuario (UDF)**. Con ellas, puedes empaquetar funciones nativas de Python y utilizarlas directamente sobre las columnas de un DataFrame de manera similar a cualquier otra función integrada.

---

## El Proceso de Creación de una UDF

Llevar una función nativa de Python al entorno distribuido de Spark consta de **tres pasos principales**:

### Paso 1: Definir la función nativa en Python
Primero, escribimos la lógica matemática o de strings utilizando código estándar de Python. En este caso, la función para calcular el cubo de un número.


In [3]:
!pip install findspark

import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [4]:
def cubo(n):
    return n * n * n

In [5]:
from pyspark.sql.types import LongType

In [25]:
#Udf primera alternativa - numerica

spark.udf.register('cubo', cubo, LongType())

<function __main__.cubo(n)>

In [8]:
spark.range(1,10).createOrReplaceTempView('df_temp')

In [9]:
spark.sql("SELECT id, cubo(id) AS cubo FROM df_temp").show()


+---+----+
| id|cubo|
+---+----+
|  1|   1|
|  2|   8|
|  3|  27|
|  4|  64|
|  5| 125|
|  6| 216|
|  7| 343|
|  8| 512|
|  9| 729|
+---+----+



In [26]:
# Udf alternativa y funcion tipo String

def bienvenida(nombre):
    return ('Hola {}'.format(nombre))

In [11]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [12]:
bienvenida_udf = udf(lambda x: bienvenida(x), StringType())

In [13]:
df_nombre = spark.createDataFrame([('Jose',), ('Julia',)], ['nombre'])

In [14]:
df_nombre.show()

+------+
|nombre|
+------+
|  Jose|
| Julia|
+------+



In [15]:
from pyspark.sql.functions import col

In [16]:
df_nombre.select(
    col('nombre'),
    bienvenida_udf(col('nombre')).alias('bie_nombre')
).show()

+------+----------+
|nombre|bie_nombre|
+------+----------+
|  Jose| Hola Jose|
| Julia|Hola Julia|
+------+----------+



In [27]:
#Alternativa con @

@udf(returnType=StringType())
def mayuscula(s):
    return s.upper()

In [18]:
df_nombre.select(
    col('nombre'),
    mayuscula(col('nombre')).alias('may_nombre')
).show()

+------+----------+
|nombre|may_nombre|
+------+----------+
|  Jose|      JOSE|
| Julia|     JULIA|
+------+----------+



Uno de los problemas con las UDFs en pyspark era que tenian un rendimiendo mas lento que las UDFs en el lenguaje Scala esto se debia a que las udfs de pyspark requerian el movimiento de datos entre la maquina virtual de java y pyhton lo cual era bastante costoso para resolver esteproblema se introdujeron las UDFs de Pandas o tambien llamadas las UDFs vectorizadas

In [28]:
import pandas as pd

In [20]:
from pyspark.sql.functions import pandas_udf

In [21]:
def cubo_pandas(a: pd.Series) -> pd.Series:
    return a * a * a

In [22]:
cubo_udf = pandas_udf(cubo_pandas, returnType=LongType())

In [23]:
x = pd.Series([1, 2, 3])

print(cubo_pandas(x))

0     1
1     8
2    27
dtype: int64


In [24]:
df = spark.range(5)

df.select(
    col('id'),
    cubo_udf(col('id')).alias('cubo_pandas')
).show()

+---+-----------+
| id|cubo_pandas|
+---+-----------+
|  0|          0|
|  1|          1|
|  2|          8|
|  3|         27|
|  4|         64|
+---+-----------+

